In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import r2_score


# =========================
# 0) 설정
# =========================
XLSX_PATH = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
CHEM_SHEET = "Supplementary File S1"
SENS_SHEET = "Supplementary File S4"

# ✅ k-fold 결과 저장 루트(원하는대로 바꿔도 됨)
OUT_ROOT = Path(r"/home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# 논문 스타일 유지: 70/30 stratify split
TEST_SIZE = 0.30
OUTER_RANDOM_STATE = 0
STRATIFY_COL = "tasting_category_fine"

# 10-fold
N_SPLITS = 10
KF_RANDOM_STATE = 20260227

# PCA 설정
STANDARDIZE_BEFORE_PCA = True
EIG_TOL = 1e-12

SAVE_PLOTS = True


# =========================
# 1) 데이터 로드 + merge
# =========================
chem_df = pd.read_excel(XLSX_PATH, sheet_name=CHEM_SHEET)
sens_df = pd.read_excel(XLSX_PATH, sheet_name=SENS_SHEET)

META_COLS = ["beer", "beer_id", "tasting_category_fine"]
df = chem_df.merge(sens_df, on=META_COLS, how="inner", validate="one_to_one")

feature_cols = [c for c in chem_df.columns if c not in META_COLS]
target_cols  = [c for c in sens_df.columns if c not in META_COLS]

assert len(feature_cols) == 231
assert len(target_cols) == 50

print("data:", df.shape)
print("min class count (full):", int(df[STRATIFY_COL].value_counts().min()))


# =========================
# 2) 바깥 split: 논문처럼 stratify 70/30 유지
# =========================
outer_train_df, outer_test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=OUTER_RANDOM_STATE,
    shuffle=True,
    stratify=df[STRATIFY_COL]
)

print("outer train:", outer_train_df.shape, "outer test:", outer_test_df.shape)
print("min class count (outer train):", int(outer_train_df[STRATIFY_COL].value_counts().min()))


# =========================
# 3) PCA helper
# =========================
def pca_fit_from_train(X_train: np.ndarray, standardize: bool, eig_tol: float):
    Xtr = X_train.astype(float)
    mu = Xtr.mean(axis=0)
    Xtr_c = Xtr - mu

    if standardize:
        sigma = Xtr_c.std(axis=0, ddof=0)
        sigma_safe = sigma.copy()
        sigma_safe[sigma_safe == 0] = 1.0
        Xtr_cs = Xtr_c / sigma_safe
    else:
        sigma_safe = np.ones(Xtr.shape[1], dtype=float)
        Xtr_cs = Xtr_c

    n_train = Xtr_cs.shape[0]
    S = (Xtr_cs.T @ Xtr_cs) / (n_train - 1)

    eigvals, eigvecs = np.linalg.eigh(S)   # ascending
    idx = np.argsort(eigvals)[::-1]        # descending
    eigvals = np.clip(eigvals[idx], 0, None)
    eigvecs = eigvecs[:, idx]

    total_var = eigvals.sum()
    evr = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)

    keep_mask = eigvals > eig_tol
    V_keep = eigvecs[:, keep_mask]
    evr_keep = evr[keep_mask]

    return {
        "mu": mu, "sigma": sigma_safe,
        "eigvals": eigvals, "eigvecs": eigvecs,
        "evr": evr, "keep_mask": keep_mask,
        "V_keep": V_keep, "evr_keep": evr_keep
    }

def pca_project(X: np.ndarray, mu: np.ndarray, sigma: np.ndarray, V_keep: np.ndarray, standardize: bool):
    Xc = X.astype(float) - mu
    Xcs = Xc / sigma if standardize else Xc
    return Xcs @ V_keep

def pca_reconstruct(Z: np.ndarray, mu: np.ndarray, sigma: np.ndarray, V_keep: np.ndarray, standardize: bool):
    Xhat_cs = Z @ V_keep.T
    return Xhat_cs * sigma + mu if standardize else Xhat_cs + mu

def recon_metrics(X_true: np.ndarray, X_hat: np.ndarray):
    err = X_true - X_hat
    rmse = float(np.sqrt(np.mean(err**2)))
    mae = float(np.mean(np.abs(err)))
    max_abs_err = float(np.max(np.abs(err)))
    # flatten R2 (전체 원소 기준) — 참고용
    r2 = float(r2_score(X_true.ravel(), X_hat.ravel()))
    return {"rmse": rmse, "mae": mae, "max_abs_err": max_abs_err, "r2_flat": r2}

def worst_zscore_report(X: np.ndarray, mu: np.ndarray, sigma: np.ndarray, meta_df: pd.DataFrame, topn=10):
    X = X.astype(float)
    Z = (X - mu) / sigma
    absmax_per_sample = np.max(np.abs(Z), axis=1)
    worst_idx = np.argsort(absmax_per_sample)[::-1][:topn]

    rows = []
    for i in worst_idx:
        j = int(np.argmax(np.abs(Z[i])))
        rows.append({
            "row_index": int(i),
            "beer": meta_df.iloc[i]["beer"],
            "beer_id": meta_df.iloc[i]["beer_id"],
            "style": meta_df.iloc[i][STRATIFY_COL],
            "max_abs_z": float(absmax_per_sample[i]),
            "worst_feature": feature_cols[j],
            "raw_value": float(X[i, j]),
            "mu": float(mu[j]),
            "sigma": float(sigma[j]),
            "z": float(Z[i, j]),
        })
    return pd.DataFrame(rows)


# =========================
# 4) train 내부 10-fold splitter 결정
# =========================
min_count_train = int(outer_train_df[STRATIFY_COL].value_counts().min())

if min_count_train >= N_SPLITS:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=KF_RANDOM_STATE)
    split_iter = splitter.split(outer_train_df, outer_train_df[STRATIFY_COL])
    splitter_name = "StratifiedKFold"
else:
    # ⚠️ 10-fold stratify 불가능 → KFold로 fallback
    splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=KF_RANDOM_STATE)
    split_iter = splitter.split(outer_train_df)
    splitter_name = "KFold"
    print(f"⚠️ StratifiedKFold(n_splits={N_SPLITS}) 불가: outer_train 최소 클래스={min_count_train}")
    print("⚠️ train 내부 fold는 KFold(셔플)로 진행합니다. (fold별 style 분포는 저장됨)")


# =========================
# 5) 10-fold 실행
# =========================
summary_rows = []

# outer test는 고정(모든 fold에서 같은 outer test로 복원도 같이 봄)
X_outer_test = outer_test_df[feature_cols].copy()
# impute는 fold train mean으로 해야 하므로 여기서는 아직 fillna 하지 않음

outer_test_meta = outer_test_df[META_COLS].reset_index(drop=True)

for fold_idx, (tr_idx, va_idx) in enumerate(split_iter, start=1):
    fold_dir = OUT_ROOT / str(fold_idx) / "종합"
    fold_dir.mkdir(parents=True, exist_ok=True)

    fold_train_df = outer_train_df.iloc[tr_idx].copy()
    fold_val_df   = outer_train_df.iloc[va_idx].copy()

    # --- split assignment 저장(추적용) ---
    assign = pd.concat([
        fold_train_df[META_COLS].assign(part="fold_train", fold=fold_idx),
        fold_val_df[META_COLS].assign(part="fold_val", fold=fold_idx),
        outer_test_df[META_COLS].assign(part="outer_test", fold=fold_idx),
    ], axis=0, ignore_index=True)
    assign.to_csv(fold_dir / "split_assignment.csv", index=False, encoding="utf-8-sig")

    # --- style 분포 저장 ---
    counts = pd.concat([
        fold_train_df[STRATIFY_COL].value_counts().rename("fold_train"),
        fold_val_df[STRATIFY_COL].value_counts().rename("fold_val"),
        outer_test_df[STRATIFY_COL].value_counts().rename("outer_test"),
    ], axis=1).fillna(0).astype(int)
    counts.to_csv(fold_dir / "style_counts.csv", encoding="utf-8-sig")

    # --- X 생성 + 결측치 처리(train fold 평균) ---
    Xtr = fold_train_df[feature_cols].copy()
    Xva = fold_val_df[feature_cols].copy()
    Xte = outer_test_df[feature_cols].copy()

    impute_means = Xtr.mean(axis=0)
    Xtr = Xtr.fillna(impute_means)
    Xva = Xva.fillna(impute_means)
    Xte = Xte.fillna(impute_means)

    Xtr_np = Xtr.values
    Xva_np = Xva.values
    Xte_np = Xte.values

    # --- PCA fit: fold_train만 ---
    pca = pca_fit_from_train(Xtr_np, STANDARDIZE_BEFORE_PCA, EIG_TOL)
    mu, sigma = pca["mu"], pca["sigma"]
    eigvals, eigvecs, evr = pca["eigvals"], pca["eigvecs"], pca["evr"]
    keep_mask, V_keep, evr_keep = pca["keep_mask"], pca["V_keep"], pca["evr_keep"]
    k_keep = int(np.sum(keep_mask))

    # PCA 축 저장
    np.savez(
        fold_dir / "pca_axes_full_231.npz",
        feature_cols=np.array(feature_cols, dtype=object),
        mu=mu, sigma=sigma,
        standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
        eig_tol=np.array([EIG_TOL]),
        eigvals=eigvals, eigvecs=eigvecs, evr=evr,
        splitter=np.array([splitter_name], dtype=object),
        fold=np.array([fold_idx]),
    )
    np.savez(
        fold_dir / "pca_axes_kept_nonzero.npz",
        feature_cols=np.array(feature_cols, dtype=object),
        mu=mu, sigma=sigma,
        standardize_before_pca=np.array([STANDARDIZE_BEFORE_PCA]),
        eig_tol=np.array([EIG_TOL]),
        keep_mask=keep_mask,
        kept_full_indices=np.where(keep_mask)[0],
        V_keep=V_keep,
        evr_keep=evr_keep,
        splitter=np.array([splitter_name], dtype=object),
        fold=np.array([fold_idx]),
    )

    # EVR 저장
    ev_df = pd.DataFrame({
        "pc_index_1based_full": np.arange(1, len(eigvals)+1),
        "eigenvalue": eigvals,
        "explained_variance_ratio": evr,
        "explained_variance_ratio_percent": evr * 100.0,
        "kept_nonzero": keep_mask
    })
    ev_df.to_csv(fold_dir / "explained_variance_ratio_all_pcs.csv", index=False, encoding="utf-8-sig")

    if SAVE_PLOTS:
        plt.figure(figsize=(9,4))
        plt.plot(evr*100, marker="o", linewidth=1)
        plt.title(f"EVR per PC (%) — fold {fold_idx}")
        plt.xlabel("PC index")
        plt.ylabel("EVR (%)")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(fold_dir / "evr_per_pc.png", dpi=160)
        plt.close()

        plt.figure(figsize=(9,4))
        plt.plot(np.cumsum(evr)*100, marker="o", linewidth=1)
        plt.title(f"Cumulative EVR (%) — fold {fold_idx}")
        plt.xlabel("PC index")
        plt.ylabel("Cumulative EVR (%)")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(fold_dir / "cumulative_evr.png", dpi=160)
        plt.close()

    # --- 투영/복원 + 복원력 ---
    Ztr = pca_project(Xtr_np, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Zva = pca_project(Xva_np, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Zte = pca_project(Xte_np, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)

    Xtr_hat = pca_reconstruct(Ztr, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Xva_hat = pca_reconstruct(Zva, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)
    Xte_hat = pca_reconstruct(Zte, mu, sigma, V_keep, STANDARDIZE_BEFORE_PCA)

    m_tr = recon_metrics(Xtr_np, Xtr_hat)
    m_va = recon_metrics(Xva_np, Xva_hat)
    m_te = recon_metrics(Xte_np, Xte_hat)

    recon = pd.DataFrame([{
        "fold": fold_idx,
        "splitter": splitter_name,
        "n_fold_train": Xtr_np.shape[0],
        "n_fold_val": Xva_np.shape[0],
        "n_outer_test": Xte_np.shape[0],
        "p": Xtr_np.shape[1],
        "k_keep": k_keep,
        "standardize_before_pca": STANDARDIZE_BEFORE_PCA,
        "eig_tol": EIG_TOL,

        "fold_train_recon_rmse": m_tr["rmse"],
        "fold_train_recon_r2_flat": m_tr["r2_flat"],
        "fold_val_recon_rmse": m_va["rmse"],
        "fold_val_recon_r2_flat": m_va["r2_flat"],
        "outer_test_recon_rmse": m_te["rmse"],
        "outer_test_recon_r2_flat": m_te["r2_flat"],

        "fold_val_max_abs_err": m_va["max_abs_err"],
        "outer_test_max_abs_err": m_te["max_abs_err"],
    }])
    recon.to_csv(fold_dir / "reconstruction_metrics.csv", index=False, encoding="utf-8-sig")

    # outlier report(복원이 터지는 원인 후보)
    fold_val_meta = fold_val_df[META_COLS].reset_index(drop=True)
    worst_val = worst_zscore_report(Xva_np, mu, sigma, fold_val_meta, topn=10)
    worst_te  = worst_zscore_report(Xte_np, mu, sigma, outer_test_meta, topn=10)
    worst_val.to_csv(fold_dir / "val_outlier_report_top10.csv", index=False, encoding="utf-8-sig")
    worst_te.to_csv(fold_dir / "outer_test_outlier_report_top10.csv", index=False, encoding="utf-8-sig")

    summary_rows.append(recon.iloc[0].to_dict())

    print(f"[fold {fold_idx}/{N_SPLITS}] saved -> {fold_dir.parent}")

# 전체 요약
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_ROOT / "summary_kfold10_reconstruction.csv", index=False, encoding="utf-8-sig")
print("✅ saved:", OUT_ROOT / "summary_kfold10_reconstruction.csv")

display(summary[[
    "fold","splitter","k_keep",
    "fold_val_recon_rmse","outer_test_recon_rmse",
    "outer_test_max_abs_err"
]].sort_values("outer_test_recon_rmse", ascending=False).head(10))

print("\nMean±Std (val recon rmse):",
      summary["fold_val_recon_rmse"].mean(), "±", summary["fold_val_recon_rmse"].std(ddof=1))
print("Mean±Std (outer_test recon rmse):",
      summary["outer_test_recon_rmse"].mean(), "±", summary["outer_test_recon_rmse"].std(ddof=1))

data: (250, 284)
min class count (full): 3
outer train: (175, 284) outer test: (75, 284)
min class count (outer train): 2
⚠️ StratifiedKFold(n_splits=10) 불가: outer_train 최소 클래스=2
⚠️ train 내부 fold는 KFold(셔플)로 진행합니다. (fold별 style 분포는 저장됨)
[fold 1/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/1
[fold 2/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/2
[fold 3/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/3
[fold 4/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/4
[fold 5/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/5
[fold 6/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/6
[fold 7/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/7
[fold 8/10] saved -> /home/a202192020/맥주데이터실험/맥주데이터실험/pca_origin_fold_0227/k_fold/output/8
[fold 9/10] saved -> /home/a2021920

,fold,splitter,k_keep,fold_val_recon_rmse,outer_test_recon_rmse,outer_test_max_abs_err
3,4,KFold,156,14.405452,1063.287623,138709.207504
2,3,KFold,156,15.745226,876.102323,112775.180976
6,7,KFold,157,40.259155,866.668107,109382.871779
8,9,KFold,157,23.448724,844.257856,80235.613249
7,8,KFold,157,12.196525,588.676110,75557.730849
9,10,KFold,157,8.764827,544.154594,66081.574312
0,1,KFold,156,20.696019,498.278289,56191.737378
4,5,KFold,156,1074.638251,309.556380,39920.775044
5,6,KFold,157,49.934368,235.310694,24405.920679
1,2,KFold,156,42.169828,185.488093,22505.908003



Mean±Std (val recon rmse): 130.2258374086326 ± 332.13307111167325
Mean±Std (outer_test recon rmse): 601.178006791975 ± 302.6545465205831
